# DDSP Timbre Transfer — All 5 Pre-trained Models

**Music Style Transfer Project** — Yotam & Gal (Phase 4)

This notebook transfers the timbre of a **monophonic** input clip using
Google's [DDSP](https://github.com/magenta/ddsp) (Differentiable Digital Signal Processing).

It runs **all 5 pre-trained instrument models**:
| Model | Instrument |
|-------|------------|
| Violin | Bowed string |
| Flute | Woodwind |
| Flute2 | Woodwind (variant) |
| Trumpet | Brass |
| Tenor Saxophone | Brass/woodwind |

### How it works
1. Extract **pitch (f0)** and **loudness** from your input audio
2. Adjust these features to match each instrument's training distribution
3. Feed into the DDSP autoencoder → synthesize audio with new timbre

### Setup
1. Set runtime to **GPU → T4** (Runtime → Change runtime type)
2. Upload a **monophonic** WAV to Google Drive: `My Drive/MusicProject_Colab/`
   - A Demucs-separated stem (vocals, bass, etc.) works well
   - Or any solo instrument / voice recording
3. Run all cells in order

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Runtime → Change runtime type → T4 GPU")

## 2. Install DDSP

DDSP v1.6.5 (TensorFlow-based). This takes ~1-2 minutes.

In [ ]:
# Install DDSP without automatic dependency resolution
# (tensorflow-addons and hmmlearn are incompatible with Colab's Python 3.12)
!pip install --no-deps ddsp==1.6.5
!pip install -q tensorflow gin-config crepe pydub mir_eval scipy
print('DDSP installed')

In [ ]:
# Create fake modules for DDSP's unused dependencies
import types, sys, os

fake_modules = [
    'tensorflow_addons', 'hypertune',
    'tensorflowjs', 'tensorflow_datasets', 'tflite_support',
    'google.cloud', 'google.cloud.storage',
    'note_seq', 'note_seq.sequences_lib',
    'note_seq.protobuf', 'note_seq.protobuf.music_pb2',
]

for mod_name in fake_modules:
    if mod_name not in sys.modules:
        mod = types.ModuleType(mod_name)
        sys.modules[mod_name] = mod
        parts = mod_name.rsplit('.', 1)
        if len(parts) == 2 and parts[0] in sys.modules:
            setattr(sys.modules[parts[0]], parts[1], mod)

# Fix Python 3.12: collections.Iterable removed
import collections
import collections.abc
collections.Iterable = collections.abc.Iterable
collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping
collections.MutableSequence = collections.abc.MutableSequence
collections.Sequence = collections.abc.Sequence

# Fix TF 2.19+ / Keras 3: force legacy Keras 2 so GRUCell checkpoint loading works
# DDSP pre-trained checkpoints were saved with old Keras — must use legacy API
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import warnings
warnings.filterwarnings("ignore")

import ddsp
import ddsp.training
from ddsp.colab.colab_utils import (
    auto_tune, get_tuning_factor, download,
    play, record, specplot, upload,
    DEFAULT_SAMPLE_RATE
)
from ddsp.training.postprocessing import detect_notes, fit_quantile_transform
import gin
import tensorflow.compat.v2 as tf
import numpy as np
import librosa
import matplotlib.pyplot as plt
import time, copy, pickle

SAMPLE_RATE = DEFAULT_SAMPLE_RATE  # 16000
print(f"DDSP {ddsp.__version__} loaded!")
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {getattr(tf.keras, '__version__', 'built-in')}")
print(f"Sample rate: {SAMPLE_RATE} Hz")

## 3. Mount Google Drive & Load Audio

Upload a **monophonic** WAV or MP3 to `My Drive / MusicProject_Colab /`.

Good input sources:
- A Demucs-separated stem (vocals.wav, bass.wav, other.wav)
- Any solo instrument recording
- **Not** a full mix — DDSP needs single-instrument audio

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_FOLDER = "/content/drive/MyDrive/MusicProject_Colab"

if not os.path.isdir(DRIVE_FOLDER):
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    print(f"Created {DRIVE_FOLDER} — upload your WAV there and re-run")
else:
    # List available audio files
    audio_exts = {".wav", ".mp3", ".flac", ".ogg"}
    audio_files = [f for f in os.listdir(DRIVE_FOLDER)
                   if Path(f).suffix.lower() in audio_exts]
    print(f"Audio files in MusicProject_Colab/:")
    for i, f in enumerate(audio_files):
        size = os.path.getsize(os.path.join(DRIVE_FOLDER, f)) / 1e6
        print(f"  [{i}] {f} ({size:.1f} MB)")
    if not audio_files:
        print("  No audio files found! Upload a WAV and re-run.")

### Select input file

Change `FILE_INDEX` below to pick which file to use (from the list above).

In [ ]:
FILE_INDEX = 0  # ← Change this to select a different file

input_file = os.path.join(DRIVE_FOLDER, audio_files[FILE_INDEX])
input_name = Path(input_file).stem
print(f"Loading: {input_name}")
print(f"Path: {input_file}")

# Load audio at DDSP's native 16kHz
audio_orig, _ = librosa.load(input_file, sr=SAMPLE_RATE, mono=True)
print(f"Duration: {len(audio_orig)/SAMPLE_RATE:.1f}s")
print(f"Samples: {len(audio_orig):,}")

# DDSP expects shape (1, n_samples)
audio = audio_orig[np.newaxis, :]

# Visualize
specplot(audio)
plt.title(f"Input: {input_name}")
plt.show()
play(audio)

## 4. Extract Audio Features (CREPE Pitch + Loudness)

DDSP extracts:
- **f0** (fundamental frequency) using CREPE neural pitch detector
- **loudness** in dB
- **f0_confidence** — how certain the pitch estimate is

This takes ~30-60 seconds depending on audio length.

In [ ]:
# Reset CREPE to avoid memory issues
ddsp.spectral_ops.reset_crepe()

# Compute features
start_time = time.time()
audio_features = ddsp.training.metrics.compute_audio_features(audio)
audio_features['loudness_db'] = audio_features['loudness_db'].astype(np.float32)
elapsed = time.time() - start_time
print(f"Feature extraction took {elapsed:.1f}s")

# Plot features
TRIM = -15  # Trim last few unstable frames
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(12, 8))

axes[0].plot(audio_features['loudness_db'][:TRIM])
axes[0].set_ylabel('Loudness (dB)')

axes[1].plot(librosa.hz_to_midi(audio_features['f0_hz'][:TRIM]))
axes[1].set_ylabel('Pitch (MIDI)')

axes[2].plot(audio_features['f0_confidence'][:TRIM])
axes[2].set_ylabel('Pitch confidence')
axes[2].set_xlabel('Time step (frame)')

fig.suptitle(f"Audio Features: {input_name}", fontsize=14)
plt.tight_layout()
plt.show()

# Summary stats
f0_hz = audio_features['f0_hz']
conf = audio_features['f0_confidence']
voiced = conf > 0.85
print(f"\nVoiced frames: {voiced.sum()}/{len(voiced)} ({100*voiced.mean():.0f}%)")
if voiced.any():
    f0_voiced = f0_hz[voiced]
    print(f"Pitch range: {f0_voiced.min():.0f} - {f0_voiced.max():.0f} Hz "
          f"(MIDI {librosa.hz_to_midi(f0_voiced.min()):.0f} - {librosa.hz_to_midi(f0_voiced.max()):.0f})")

## 5. Run Timbre Transfer — All 5 Pre-trained Models

Each model:
1. Downloads pre-trained checkpoint from Google Cloud Storage
2. Auto-adjusts pitch register and loudness to match training data
3. Synthesizes audio with the new timbre

**Auto-tune is disabled** to preserve microtonal nuances.

This takes ~2-5 minutes for all 5 models.

In [ ]:
# ─── Configuration ───
MODELS = ['Violin', 'Flute', 'Flute2', 'Trumpet', 'Tenor_Saxophone']
GCS_CKPT_DIR = 'gs://ddsp/models/timbre_transfer_colab/2021-07-08'
PRETRAINED_DIR = "/content/pretrained"

# Conditioning parameters (shared across all models)
THRESHOLD = 1.0    # Note detection sensitivity
QUIET_DB = 20      # Silence unvoiced regions (dB below)
AUTOTUNE = 0.0     # 0 = off, 1 = full snap to semitones (disabled for Israeli music)
PITCH_SHIFT = 0    # Manual pitch shift in octaves
LOUDNESS_SHIFT = 0 # Manual loudness shift in dB

results = {}

for model_name in MODELS:
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    # ── Download model checkpoint ──
    !rm -rf {PRETRAINED_DIR} 2>/dev/null
    !mkdir -p {PRETRAINED_DIR}
    model_gcs = os.path.join(GCS_CKPT_DIR, f'solo_{model_name.lower()}_ckpt')
    !gsutil cp {model_gcs}/* {PRETRAINED_DIR} 2>/dev/null

    model_dir = PRETRAINED_DIR
    gin_file = os.path.join(model_dir, 'operative_config-0.gin')

    # ── Load dataset statistics (for auto-adjustment) ──
    DATASET_STATS = None
    stats_file = os.path.join(model_dir, 'dataset_statistics.pkl')
    try:
        if tf.io.gfile.exists(stats_file):
            with tf.io.gfile.GFile(stats_file, 'rb') as f:
                DATASET_STATS = pickle.load(f)
    except Exception as e:
        print(f"  Warning: Could not load stats: {e}")

    # ── Configure model ──
    with gin.unlock_config():
        gin.clear_config()
        gin.parse_config_file(gin_file, skip_unknown=True)

    time_steps_train = gin.query_parameter('F0LoudnessPreprocessor.time_steps')
    n_samples_train = gin.query_parameter('Harmonic.n_samples')
    hop_size = int(n_samples_train / time_steps_train)

    time_steps = int(audio.shape[1] / hop_size)
    n_samples = time_steps * hop_size

    gin_params = [
        f'Harmonic.n_samples = {n_samples}',
        f'FilteredNoise.n_samples = {n_samples}',
        f'F0LoudnessPreprocessor.time_steps = {time_steps}',
        'oscillator_bank.use_angular_cumsum = True',
    ]
    with gin.unlock_config():
        gin.parse_config(gin_params)

    # ── Trim features to match model's expected length ──
    af = copy.deepcopy(audio_features)
    for key in ['f0_hz', 'f0_confidence', 'loudness_db']:
        af[key] = af[key][:time_steps]
    af['audio'] = af['audio'][:, :n_samples]

    # ── Auto-adjust conditioning ──
    if DATASET_STATS is not None:
        mask_on, note_on_value = detect_notes(
            af['loudness_db'], af['f0_confidence'], THRESHOLD)

        if np.any(mask_on):
            # Shift pitch register to match target instrument
            target_mean_pitch = DATASET_STATS['mean_pitch']
            pitch = ddsp.core.hz_to_midi(af['f0_hz'])
            mean_pitch = np.mean(pitch[mask_on])
            p_diff = target_mean_pitch - mean_pitch
            p_diff_octave = p_diff / 12.0
            round_fn = np.floor if p_diff_octave > 1.5 else np.ceil
            p_diff_octave = round_fn(p_diff_octave)
            af['f0_hz'] *= 2.0 ** p_diff_octave
            af['f0_hz'] = np.clip(af['f0_hz'], 0.0, librosa.midi_to_hz(110.0))

            # Quantile-transform loudness to match training distribution
            _, loudness_norm = fit_quantile_transform(
                af['loudness_db'], mask_on,
                inv_quantile=DATASET_STATS['quantile_transform'])
            mask_off = np.logical_not(mask_on)
            loudness_norm[mask_off] -= QUIET_DB * (1.0 - note_on_value[mask_off][:, np.newaxis])
            af['loudness_db'] = np.reshape(loudness_norm, af['loudness_db'].shape)

            # Auto-tune (disabled by default for Israeli music)
            if AUTOTUNE > 0:
                f0_midi = np.array(ddsp.core.hz_to_midi(af['f0_hz']))
                tuning_factor = get_tuning_factor(f0_midi, af['f0_confidence'], mask_on)
                f0_midi_at = auto_tune(f0_midi, tuning_factor, mask_on, amount=AUTOTUNE)
                af['f0_hz'] = ddsp.core.midi_to_hz(f0_midi_at)

    # Manual shifts
    af['loudness_db'] += LOUDNESS_SHIFT
    af['f0_hz'] *= 2.0 ** PITCH_SHIFT
    af['f0_hz'] = np.clip(af['f0_hz'], 0.0, librosa.midi_to_hz(110.0))

    # ── Load model and synthesize ──
    ckpt_files = [f for f in tf.io.gfile.listdir(model_dir) if 'ckpt' in f]
    ckpt_name = ckpt_files[0].split('.')[0]
    ckpt = os.path.join(model_dir, ckpt_name)

    model = ddsp.training.models.Autoencoder()
    model.restore(ckpt)

    t0 = time.time()
    outputs = model(af, training=False)
    audio_gen = model.get_audio_from_outputs(outputs)
    infer_time = time.time() - t0

    # Save result
    out_path = f"/content/{input_name}_{model_name.lower()}_ddsp.wav"
    import soundfile as sf
    sf.write(out_path, audio_gen[0].numpy(), SAMPLE_RATE)
    print(f"  Inference: {infer_time:.1f}s")
    print(f"  Saved: {out_path}")

    results[model_name] = {
        'audio': audio_gen[0].numpy(),
        'output_path': out_path,
        'infer_time': infer_time
    }

    # Clean up to free memory
    del model, outputs, audio_gen
    tf.keras.backend.clear_session()
    print(f"  Done!\n")

print(f"\n✓ All {len(results)} models completed!")

## 6. Results Comparison

In [ ]:
print(f"Source: {input_name}\n")
print(f"{"Model":<20} {"Inference (s)":<15} {"Duration (s)":<15} {"File Size (KB)":<15}")
print("-" * 65)

for name, r in results.items():
    dur = len(r['audio']) / SAMPLE_RATE
    fsize = os.path.getsize(r['output_path']) / 1e3
    print(f"{name:<20} {r["infer_time"]:<15.1f} {dur:<15.1f} {fsize:<15.0f}")

## 7. Listen & Compare

Listen to the original and all 5 timbre-transferred versions.

In [ ]:
from IPython.display import Audio, display, HTML

# Original
display(HTML("<h3>Original Input</h3>"))
display(Audio(audio_orig, rate=SAMPLE_RATE))

# All models
for name, r in results.items():
    display(HTML(f"<h3>{name}</h3>"))
    display(Audio(r['audio'], rate=SAMPLE_RATE))

## 8. Spectrogram Comparison

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(n_models + 1, 1, figsize=(14, 3 * (n_models + 1)))

# Original
specplot(audio, ax=axes[0])
axes[0].set_title(f"Original: {input_name}")

# Each model
for i, (name, r) in enumerate(results.items()):
    specplot(r['audio'][np.newaxis, :], ax=axes[i + 1])
    axes[i + 1].set_title(f"{name}")

plt.tight_layout()
plt.show()

## 9. Save Outputs to Google Drive

In [ ]:
import shutil

output_dir = os.path.join(DRIVE_FOLDER, "ddsp_outputs")
os.makedirs(output_dir, exist_ok=True)

for name, r in results.items():
    dst = os.path.join(output_dir, os.path.basename(r['output_path']))
    shutil.copy2(r['output_path'], dst)
    print(f"  Saved: {os.path.basename(dst)}")

# Also save the original for easy comparison
orig_dst = os.path.join(output_dir, f"{input_name}_original.wav")
sf.write(orig_dst, audio_orig, SAMPLE_RATE)
print(f"  Saved: {os.path.basename(orig_dst)}")

print(f"\n✓ All outputs saved to: {output_dir}")
print("  Find them in Google Drive → MusicProject_Colab → ddsp_outputs/")